In [1]:
# Analysis B: the chaperone/protease census (Day 4, roadmap Part 3).
#
# Claim 1: of the 61 high-confidence ATFS-1 target genes, how many are actually
# protein-folding chaperones or QC proteases? The census (data/chaperone_protease_census.csv)
# was frozen in gate_decisions.md before this comparison was run.
import os
if os.path.basename(os.getcwd()) == "scripts":
    os.chdir("..")

import pandas as pd
from scipy.stats import mannwhitneyu

# The 61-gene regulon, same loading logic as binding.ipynb: hsp-6 and hsp-60 are
# reference rows in the source sheet, not regulon members (see gate_decisions.md,
# "Census denominator: 61, not 63").
soo_df = pd.read_excel("data/raw/ATFS1_targets_Soo.xlsx", sheet_name="Sheet1")
soo_df = soo_df.iloc[0:64].dropna(subset=["Gene name"])
soo_df = soo_df.rename(columns={
    "ATFS-1 bound \nin ChIP-seq": "soo_bound",
    "Gene sequence \nname": "seqname",
})[["Gene name", "seqname", "Score", "Score/variability", "soo_bound"]]

regulon_df = soo_df[~soo_df["Gene name"].isin(["hsp-6", "hsp-60"])].reset_index(drop=True)
if len(regulon_df) != 61:
    raise RuntimeError(f"Expected 61-gene regulon, got {len(regulon_df)}.")

print(f"ATFS-1 high-confidence regulon: {len(regulon_df)} genes")


ATFS-1 high-confidence regulon: 61 genes


In [2]:
# Rank each regulon gene on both metrics. Per the dual-metric rule (gate_decisions.md,
# Gate 0), rank 1 = strongest signal; both Score and Score/variability are computed,
# and every claim below reports both, conservative (Score/variability) first.
regulon_df["rank_score"] = regulon_df["Score"].rank(ascending=False, method="min").astype(int)
regulon_df["rank_variability"] = regulon_df["Score/variability"].rank(ascending=False, method="min").astype(int)
print("Ranked 1 (strongest) to 61 (weakest) on both metrics.")


Ranked 1 (strongest) to 61 (weakest) on both metrics.


In [3]:
# Load the frozen census and match against the regulon by name or sequence name
# (the two sources do not always use the same convention).
census_df = pd.read_csv("data/chaperone_protease_census.csv")
census_keys = set(census_df["public_name"].str.lower()) | set(census_df["seqname"].str.lower())

regulon_df["in_census"] = regulon_df.apply(
    lambda r: str(r["Gene name"]).lower() in census_keys or str(r["seqname"]).lower() in census_keys,
    axis=1
)

hits = regulon_df[regulon_df["in_census"]].sort_values("rank_score")
print(f"--- Strict census match: {len(hits)} of {len(regulon_df)} regulon genes ---")
print(hits[["Gene name", "seqname", "Score", "rank_score", "Score/variability", "rank_variability"]].to_string(index=False))


--- Strict census match: 2 of 61 regulon genes ---
Gene name  seqname       Score  rank_score Score/variability  rank_variability
   dnj-10  F22B7.5  306.680798          45         17.141011                23
   ymel-1 M03C11.5  118.842331          61          8.101193                58


In [4]:
# Permissive count: the roadmap names three borderline genes that carry a
# chaperone-adjacent domain but were excluded from the strict census on functional
# grounds (see gate_decisions.md - prx-19 is a peroxisomal import chaperone, cbp-3
# and tspo-1 do not carry a folding-chaperone domain at all). Check whether each is
# actually in the regulon, rather than assuming.
borderline_genes = ["prx-19", "cbp-3", "tspo-1"]
borderline_in_regulon = regulon_df[regulon_df["Gene name"].str.lower().isin(borderline_genes)]

print("--- Borderline genes (excluded from strict census) found in the regulon ---")
print(borderline_in_regulon[["Gene name", "seqname", "rank_score", "rank_variability"]].to_string(index=False))

strict_count = len(hits)
permissive_count = strict_count + len(borderline_in_regulon)
print(f"\nStrict count:     {strict_count} of {len(regulon_df)}")
print(f"Permissive count: {permissive_count} of {len(regulon_df)} (adds {list(borderline_in_regulon['Gene name'])})")


--- Borderline genes (excluded from strict census) found in the regulon ---
Gene name  seqname  rank_score  rank_variability
   tspo-1  C41G7.9          42                21
    cbp-3 F40F12.7          44                57
   prx-19  F54F2.8          54                20

Strict count:     2 of 61
Permissive count: 5 of 61 (adds ['tspo-1', 'cbp-3', 'prx-19'])


In [5]:
# Inferential check: are census-gene ranks systematically different from
# non-census-gene ranks within the regulon? Run separately per metric, per the
# dual-metric rule. With only 2 genes in the strict census this test has very
# limited power - it is a supporting check, not the headline result. The count
# itself is the metric-independent, primary finding (roadmap Part 0).
print("--- Mann-Whitney U: census-gene ranks vs. non-census-gene ranks ---")
for label, metric in [("Score (uncorrected)", "rank_score"), ("Score/variability (corrected)", "rank_variability")]:
    a = regulon_df.loc[regulon_df["in_census"], metric]
    b = regulon_df.loc[~regulon_df["in_census"], metric]
    stat, p = mannwhitneyu(a, b, alternative="two-sided")
    print(f"{label}: census-gene ranks = {sorted(a.tolist())}, "
          f"median non-census rank = {b.median():.1f}, p = {p:.4f} (n={len(a)}, underpowered)")

print(f"\n=== Headline (Claim 1) ===")
print(f"{strict_count} of {len(regulon_df)} annotated mitochondrial/cytosolic chaperones and QC proteases")
print(f"appear among ATFS-1 targets: {sorted(hits['Gene name'].tolist())},")
print(f"at rank {sorted(hits['rank_score'].tolist())} (uncorrected) / "
      f"{sorted(hits['rank_variability'].tolist())} (variability-corrected) of {len(regulon_df)}.")


--- Mann-Whitney U: census-gene ranks vs. non-census-gene ranks ---
Score (uncorrected): census-gene ranks = [45, 61], median non-census rank = 30.0, p = 0.0787 (n=2, underpowered)
Score/variability (corrected): census-gene ranks = [23, 58], median non-census rank = 31.0, p = 0.4820 (n=2, underpowered)

=== Headline (Claim 1) ===
2 of 61 annotated mitochondrial/cytosolic chaperones and QC proteases
appear among ATFS-1 targets: ['dnj-10', 'ymel-1'],
at rank [45, 61] (uncorrected) / [23, 58] (variability-corrected) of 61.
